In [1]:
import json
from pathlib import Path

instance_id = "astropy__astropy-12057"
src = Path("/home/mengqi/work/DaLIC_newexp/dataset/swebench_feature_with_truth_sr_scripts.jsonl").expanduser()
out = Path("/home/mengqi/work/DaLIC_newexp/tmp/manual_run").expanduser() / instance_id
out.mkdir(parents=True, exist_ok=True)

for line in src.open("r", encoding="utf-8"):
    row = json.loads(line)
    if row["instance_id"] != instance_id:
        continue
    (out / "row.json").write_text(json.dumps(row, ensure_ascii=False, indent=2), encoding="utf-8")
    (out / "A.py").write_text(row["script_A"], encoding="utf-8")
    (out / "B.py").write_text(row["script_B"], encoding="utf-8")
    (out / "patch.diff").write_text(row["patch"], encoding="utf-8")
    (out / "test_patch.diff").write_text(row["test_patch"], encoding="utf-8")
    print(out)
    break
else:
    raise SystemExit(f"instance not found: {instance_id}")

/home/mengqi/work/DaLIC_newexp/tmp/manual_run/astropy__astropy-12057


In [4]:
import json
from pathlib import Path
import docker

from swebench.harness.test_spec.test_spec import make_test_spec
from swebench.harness.docker_build import build_container, setup_logger, close_logger

instance_id = "astropy__astropy-12057"
row_path = Path(f"/home/mengqi/work/DaLIC_newexp/tmp/manual_run/{instance_id}/row.json").expanduser()
row = json.loads(row_path.read_text(encoding="utf-8"))

client = docker.from_env()
spec = make_test_spec(row, namespace="swebench")

log_dir = Path(f"/home/mengqi/work/DaLIC_newexp/tmp/manual_run/{instance_id}/build_logs").expanduser()
log_dir.mkdir(parents=True, exist_ok=True)
logger = setup_logger(instance_id, log_dir / "prepare_container.log")

container = build_container(
    test_spec=spec,
    client=client,
    run_id="manual-ab",
    logger=logger,
    nocache=False,
    force_rebuild=False,
)
close_logger(logger)

print("instance_image_key =", spec.instance_image_key)
print("container_name     =", container.name)
print("container_id       =", container.id)

instance_image_key = swebench/sweb.eval.x86_64.astropy_1776_astropy-12057:latest
container_name     = sweb.eval.astropy__astropy-12057.manual-ab
container_id       = 2f3a836c75ee1ddf83ab9c9c82607a10f11b4a201778fbe3aa13d1d63f4e46b8


In [ ]:
docker cp /home/mengqi/work/DaLIC_newexp/tmp/manual_run/astropy__astropy-12057/A.py sweb.eval.astropy__astropy-12057.manual-ab:/tmp/A.py
docker cp /home/mengqi/work/DaLIC_newexp/tmp/manual_run/astropy__astropy-12057/B.py sweb.eval.astropy__astropy-12057.manual-ab:/tmp/B.py
docker cp /home/mengqi/work/DaLIC_newexp/tmp/manual_run/astropy__astropy-12057/patch.diff sweb.eval.astropy__astropy-12057.manual-ab:/tmp/patch.diff
docker cp /home/mengqi/work/DaLIC_newexp/tmp/manual_run/astropy__astropy-12057/test_patch.diff sweb.eval.astropy__astropy-12057.manual-ab:/tmp/test_patch.diff

docker exec -u root -w /testbed sweb.eval.astropy__astropy-12057.manual-ab git apply -v /tmp/patch.diff
docker exec -u root -w /testbed sweb.eval.astropy__astropy-12057.manual-ab git apply -v /tmp/test_patch.diff

docker exec -u root -w /testbed sweb.eval.astropy__astropy-12057.manual-ab git -c core.fileMode=false diff --stat

docker exec -u root sweb.eval.astropy__astropy-12057.manual-ab bash -lc \
'source /opt/miniconda3/etc/profile.d/conda.sh && \
 conda activate testbed && \
 cd /testbed && \
 python /tmp/A.py'

docker exec -u root sweb.eval.astropy__astropy-12057.manual-ab bash -lc \
'source /opt/miniconda3/etc/profile.d/conda.sh && \
 conda activate testbed && \
 cd /testbed && \
 python /tmp/B.py'